# ch04 Bonus 06：门控 DeltaNet（Gated DeltaNet）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/08_deltanet
> **参考真实模型**：NVIDIA Nemotron-Nano / 部分 2024-2025 新模型

## 一句话

用**线性注意力 + delta 更新规则 + 门控**替代标准 softmax 注意力，把 O(n²) 的注意力压成 O(n) 递推，兼顾效率与质量。

## 背景：线性注意力与 delta rule

标准 softmax 注意力每个 token 要和所有历史 token 算分数，是 O(n²)。线性注意力去掉 softmax，让 K、V 可以累积成一个**状态矩阵 S**，每个新 token 只需更新 S：

```
标准注意力:  O(n²)  —— 每对 token 都要算
线性注意力:  O(n)   —— 维护状态矩阵 S，逐步更新
```

**Delta rule** 是一种状态更新方式（类似 DeltaNet/NPLM）：

```
S_t = α_t · S_{t-1} + β_t · (v_t − S_{t-1}·k_t) ⊗ k_t
```

- `α_t`（门控）：遗忘系数，衰减历史状态
- `β_t`：学习率，控制新信息写入强度
- `(v_t − S·k_t)`：用当前状态预测 v 的误差，只修正误差部分（比直接累加更精确）

> 门控让模型能动态决定每个 token 保留多少历史、写入多少新信息，这是 DeltaNet 优于朴素线性注意力的关键。

## 教学版实现

下面用逐 token 循环 + einsum 演示 delta rule 的递推过程。真实实现会用 chunkwise（分块并行）加速。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedDeltaNetSimplified(nn.Module):
    """门控 DeltaNet 教学版：逐 token 递推状态矩阵 S。"""

    def __init__(self, d_in, d_out, num_heads):
        super().__init__()
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)
        # 门控 α（遗忘）和 β（写入率），每头独立
        self.W_gate = nn.Linear(d_in, num_heads)
        self.W_beta = nn.Linear(d_in, num_heads)
        self.out_proj = nn.Linear(d_out, d_out)

    def forward(self, x):
        b, n, d = x.shape
        H, hd = self.num_heads, self.head_dim
        q = self.W_q(x).view(b, n, H, hd).transpose(1, 2)
        k = self.W_k(x).view(b, n, H, hd).transpose(1, 2)
        v = self.W_v(x).view(b, n, H, hd).transpose(1, 2)
        # 归一化 q/k（线性注意力常见做法）
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        # 状态矩阵 S：[b, H, hd, hd]，逐 token 递推
        S = torch.zeros(b, H, hd, hd, device=x.device)
        outputs = []
        for t in range(n):
            α = torch.sigmoid(self.W_gate(x[:, t]))     # [b, H] 遗忘门
            β = torch.sigmoid(self.W_beta(x[:, t]))     # [b, H] 写入门
            q_t, k_t, v_t = q[:, :, t], k[:, :, t], v[:, :, t]
            # 用当前状态预测 v：S·k
            Sk = torch.einsum('bhij,bhj->bhi', S, k_t)
            # delta 更新：只修正误差 (v - S·k)
            delta = β.unsqueeze(-1) * (v_t - Sk)
            S = α.unsqueeze(-1).unsqueeze(-1) * S + torch.einsum('bhi,bhj->bhij', delta, k_t)
            # 用更新后的状态读出输出
            out_t = torch.einsum('bhij,bhj->bhi', S, q_t)
            outputs.append(out_t)
        out = torch.stack(outputs, dim=2).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(out)

## 2. 运行并对比复杂度

In [ ]:
torch.manual_seed(123)
batch, seq, dim, n_heads = 2, 16, 768, 12
x = torch.randn(batch, seq, dim)

layer = GatedDeltaNetSimplified(dim, dim, n_heads)
out = layer(x)
print(f"DeltaNet 输出: {tuple(out.shape)}")
print(f"\n复杂度对比 (seq={seq}, head_dim=64)：")
print(f"  softmax 注意力: O(seq²·head_dim) = {seq*seq*64:,}")
print(f"  DeltaNet 递推:  O(seq·head_dim²)  = {seq*64*64:,}")
print(f"\n💡 当 seq >> head_dim 时，DeltaNet 显著省算力；长上下文尤其受益。")
print(f"   (教学版逐 token 循环较慢，真实实现用 chunkwise 分块并行。)")

---
> 📌 本 notebook 实现 delta rule 递推教学版，演示门控线性注意力思路。
> 含 chunkwise 并行的工程实现见官方 `ch04/08_deltanet`。